# NB4 · Kararı gerekçelendirmek

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---


## Bu defterde ne yapılıyor

Model bir olasılık üretiyor ama neden o olasılığı ürettiğini söylemiyor. Klinisyene
yalnızca bir sayı göstermek yetmez; gerekçesi de gerekir.

Bunun bir de hukuki karşılığı var. 16 Eylül dersinde anlatıldığı gibi, bir karar destek
yazılımının tıbbi cihaz sayılıp sayılmaması, sağlık profesyonelinin önerinin dayanağını
bağımsız olarak gözden geçirebilmesine bağlıdır. Bu defterde eklenecek katman, o
ölçütün yazılımdaki karşılığıdır.

İki düzeyde gerekçe üreteceksiniz. Genel düzeyde model neye bakıyor, tek hasta düzeyinde
bu hastada kararı ne sürükledi.


## Kurulum


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'tr'

print('Hazır.')


---

## Önceki defterden gelen kod

Bir önceki defterin sonunda toplanan bloğun tamamını aşağıdaki hücreye yapıştırınız.
İlk satırdaki `#@cdss` işaretini silmeyiniz; o blok bu defterin sonunda yeniden
toplanacak ve bir sonrakine taşınacaktır.

Blok çalıştığında önceki defterlerde yazdığınız her şey yeniden kurulur. İnternetten
veri okuyan satırlar varsa bu hücre birkaç saniye sürebilir.


In [ ]:
#@cdss onceki_defter
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol · Gelen kod


In [ ]:
kit.check_defined('model', 'X_egitim', 'X_sinama', 'y_sinama', 'olasilik', 'sonuc')


---

## Adım 1 · Model genel olarak neye bakıyor

İlk soru şudur: Model kararını verirken hangi bilgilere ağırlık veriyor?

Bunu ölçmenin yolu basittir. Bir bilgi karıştırılır, yani değerleri hastalar arasında
rastgele yer değiştirir, ve modelin başarımı ne kadar düşüyor diye bakılır. Çok düşüyorsa
model o bilgiye dayanıyor demektir.

Sonuçta yayılım değeri, ortalama kadar önemlidir. Yüz hastalık bir kümede tekrarlar
arasındaki fark, komşu bilgiler arasındaki farktan büyük olabilir. Böyle bir durumda
sıralamanın kendisi güvenilir değildir ve bir bulgu gibi sunulamaz.


### İstem 1

```
Modelin hangi bilgilere dayandığını ölçen bir işlem parçası yaz. Adı genel_gerekce olsun.

Her bilgi için şunu yap: O bilginin değerlerini hastalar arasında rastgele karıştır ve
modelin başarımının ne kadar düştüğüne bak. Karıştırmayı birden çok kez tekrarla ki
sonucun ne kadar oynadığını da görebilelim.

Sonucu bir tablo olarak geri ver. Tabloda şunlar bulunsun:
  bilgi   -> bilginin adı
  etki    -> başarımdaki ortalama düşüş
  yayilim -> tekrarlar arasındaki değişkenlik
  guvenli -> etki, yayılımın iki katından büyükse doğru, değilse yanlış

Tabloyu etkiye göre büyükten küçüğe sırala.

Sonra bu işlem parçasını sınama grubu üzerinde çalıştır, sonucu gerekce_tablosu adıyla
sakla ve ilk on beş satırını göster.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
genel_gerekce adında çalıştırılabilir bir işlem parçası olmalı ve bir tablo döndürmeli.
gerekce_tablosu adında bir tablo hazır olmalı ve bilgi, etki, yayilim, guvenli
sütunlarını içermeli.
```


In [ ]:
#@cdss genel_gerekce
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 1


In [ ]:
kit.check_function('genel_gerekce')
kit.check_frame(gerekce_tablosu, name='gerekce_tablosu',
                required=['bilgi', 'etki', 'yayilim', 'guvenli'])


In [ ]:
kararsiz = int((~gerekce_tablosu['guvenli'].astype(bool)).sum())
print(f'Sıralaması güvenilir olmayan bilgi sayısı: {kararsiz} / {len(gerekce_tablosu)}')
print('Bu sayı yüksekse bilgi sıralamasını bir bulgu olarak sunmayınız.')


### Python notu · Döngü ve liste

Gelen kodda `for bilgi in ...:` biçiminde satırlar göreceksiniz. Bu bir **döngüdür**:
Aynı işi bir listedeki her öğe için tekrarlar. Burada her bilgi için karıştırma işlemi
tekrarlanıyor.

Döngünün altındaki satırların içeriden başlaması, yani girintili yazılması rastgele
değildir. Python'da hangi satırların döngüye ait olduğu girintiyle belirlenir; başka
dillerdeki süslü parantezlerin yerini tutar.

**Liste** ise sıralı bir topluluktur: `[0.12, 0.08, 0.31]` gibi. Sözlükten farkı ada
değil sıraya göre erişilmesidir. Karıştırma tekrarlarının sonuçları önce bir listede
toplanır, sonra ortalaması ve yayılımı hesaplanır.


---

## Adım 2 · Tek bir hastada kararı ne sürükledi

Genel gerekçe modeli anlatır, tek hasta gerekçesi ise o hastanın önünüze gelme sebebini
anlatır. Klinisyenin ihtiyaç duyduğu ikincisidir.

Bu adımda üç hastaya bakacaksınız: Modelin doğru yakaladığı bir hasta, boşuna uyardığı
bir hasta ve kaçırdığı bir hasta.

**Asıl önemli olan boşuna uyardığı hastadır.** Yanlış bir tahmini makul gösteren bir
gerekçe, açıklanabilirliğin hatayı akladığı yerdir. Bunu bir kez görmek, kavramın
tanımını okumaktan daha öğreticidir.


### İstem 2

```
Tek bir hasta için modelin kararının gerekçesini üreten bir işlem parçası yaz. Adı
hasta_gerekcesi olsun; kendisine bir hastanın bilgilerini alsın.

Bu hastada hangi bilgilerin kararı yukarı, hangilerinin aşağı çektiğini hesapla. Her
bilginin katkısını ve o hastadaki gerçek değerini birlikte ver.

Sonucu bir tablo olarak geri ver. Tabloda bilgi, deger ve katki sütunları bulunsun;
katkının büyüklüğüne göre sırala.

Ayrıca üç hastayı bulan kısa bir kod yaz: Modelin doğru yakaladığı bir hasta, boşuna
uyardığı bir hasta ve kaçırdığı bir hasta. Bunları vakalar adında bir sözlükte sakla;
anahtarları dogru_uyari, bosuna_uyari ve kacirilan olsun, değerleri satır numarası
olsun. Bir vaka bulunamazsa değeri None olsun.

BİÇİM
Tek bir Python hücresi yaz. Satırların yanına Türkçe açıklama ekle. Kısa ve
okunabilir yaz; kodu Python bilmeyen biri takip edebilmeli. Hazır kısayollar yerine
adımları açıkça göster. Kodun altına, ne yaptığını üç cümleyle sade bir dille özetle.

BEKLENEN SONUÇ
hasta_gerekcesi adında çalıştırılabilir bir işlem parçası olmalı ve bir tablo döndürmeli.
vakalar adında bir sözlük hazır olmalı; dogru_uyari, bosuna_uyari, kacirilan
anahtarlarını içermeli.
```


In [ ]:
#@cdss hasta_gerekcesi
# Ürettiğiniz kodu bu satırın altına yapıştırınız.


### Kontrol 2


In [ ]:
kit.check_function('hasta_gerekcesi')

for ad in ['dogru_uyari', 'bosuna_uyari', 'kacirilan']:
    v = vakalar.get(ad)
    print(f'{ad:<14}', 'bulunamadı' if v is None else f'satır {v}')


### Üç vakaya bakınız

Aşağıdaki hazır hücre üç hastanın gerekçesini yan yana gösterir. Çalıştırdıktan sonra
bölümün altındaki soruları cevaplayınız.


In [ ]:
for ad in ['dogru_uyari', 'bosuna_uyari', 'kacirilan']:
    satir = vakalar.get(ad)
    if satir is None:
        continue
    print('=' * 60)
    print(ad.upper(), '· olasılık', round(float(olasilik[satir]), 3))
    print('=' * 60)
    print(hasta_gerekcesi(X_sinama[satir:satir+1]).head(8).to_string(index=False))
    print()


### Gerekçenin eleştirisi

Üç soruyu kendiniz cevaplayınız, sonra aynı soruları yapay zekâ aracına yöneltip
cevaplarını karşılaştırınız.

**Yüksek katkılı bilgilerden biri klinik bir işaret yerine kaydın yapılış biçiminden
kaynaklanıyor olabilir mi?** Ölçüm sayısı gibi sütunlar buna adaydır. Bir hastadan ilk
altı saatte çok sayıda ölçüm alınmış olması, o hastanın zaten ağır kabul edildiğinin
göstergesi olabilir. Bu klinik bir bulgu değil, bakım yoğunluğunun bir yansımasıdır.

**Boşuna uyarılan hastanın gerekçesini okuyan bir klinisyen ikna olur muydu?** Olursa bu
bir başarı değil sorundur; sistem yanlış kararını da ikna edici biçimde savunabiliyor
demektir.

**Bu katkılardan hangileri nedensellik sanılabilir?** Düşük tansiyonun yüksek katkı
vermesi, tansiyonu yükseltmenin yatış süresini kısaltacağı anlamına gelmez. Gerekçe bir
ilişki gösterir, bir sebep göstermez.


---

## Defter sonu · Kodun toplanması

Aşağıdaki hücre önceki defterlerden taşıdığınız kodla bu defterde eklediklerinizi tek
bir blok hâlinde toplar. Çıkan bloğun tamamını kopyalayınız; NB5 defterinin ilk
hücresine yapıştıracaksınız.

Blok ayrıca `cdss_nb4.py` adıyla kaydedilir. Colab oturumu kapandığında bu dosya silinir, bu
nedenle bloğu kendi bilgisayarınızda bir metin dosyasına da kopyalayınız.


In [ ]:
kod = kit.export(save_as='cdss_nb4.py')


## Bu defterde ne yapıldı

Sisteme iki gerekçe katmanı eklendi: Genel ve hasta düzeyinde.

Gerekçenin ne verip ne vermediği de görüldü. Verdiği: Hata avı, alt grup denetimi ve
klinisyenle konuşulabilecek somut bir nesne. Vermediği: Nedensellik ve doğruluk
güvencesi.

NB5'te sisteme güvenlik bariyerleri eklenecek ve bir uyumluluk raporu üretilecektir.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelir ve Türkiye'deki bir yoğun
bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
